## Cell 1 - Imports

In [18]:
# Import the libraries used in this notebook section.
import pandas as pd
import numpy as np
import os

## Cell 2 - Load the TED dataset

In [2]:
# Set up project paths so files can be read and outputs can be organised consistently.
file_path = "../data/raw/ted_most_recent_50000.csv"

# Load the dataset needed for the next analysis step.
df = pd.read_csv(file_path, low_memory=False)

# Inspect the data to confirm the structure and values look reasonable.
print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

Shape: (50000, 76)

First 5 rows:


,ID_NOTICE_CAN,TED_NOTICE_URL,YEAR,ID_TYPE,DT_DISPATCH,XSD_VERSION,CANCELLED,CORRECTIONS,B_MULTIPLE_CAE,CAE_NAME,...,NUMBER_TENDERS_SME,NUMBER_TENDERS_OTHER_EU,NUMBER_TENDERS_NON_EU,NUMBER_OFFERS_ELECTR,AWARD_EST_VALUE_EURO,AWARD_VALUE_EURO,AWARD_VALUE_EURO_FIN_1,B_SUBCONTRACTED,DT_AWARD,source_year
0,20232,ted.europa.eu/udl?uri=TED:NOTICE:2-2023:TEXT:E...,2023,3,28/12/22,R209.S5,0,0,N,Österreichische Gesundheitskasse,...,1.0,NaN,NaN,2.0,NaN,1052600.36,1052600.36,N,28/12/22,2023
1,20234,ted.europa.eu/udl?uri=TED:NOTICE:4-2023:TEXT:E...,2023,3,28/12/22,R209.S5,0,0,N,Presidencia de la Diputación Provincial de Ali...,...,0.0,NaN,NaN,NaN,109256.2,109000.00,109000.00,N,28/12/22,2023
2,20235,ted.europa.eu/udl?uri=TED:NOTICE:5-2023:TEXT:E...,2023,3,28/12/22,R209.S5,0,0,N,Národný ústav reumatických chorôb,...,NaN,0.0,0.0,3.0,NaN,516500.00,516500.00,N,20/12/22,2023
3,20235,ted.europa.eu/udl?uri=TED:NOTICE:5-2023:TEXT:E...,2023,3,28/12/22,R209.S5,0,0,N,Národný ústav reumatických chorôb,...,NaN,0.0,0.0,3.0,NaN,564083.33,564083.33,N,20/12/22,2023
4,20235,ted.europa.eu/udl?uri=TED:NOTICE:5-2023:TEXT:E...,2023,3,28/12/22,R209.S5,0,0,N,Národný ústav reumatických chorôb,...,NaN,0.0,0.0,3.0,NaN,793500.00,793500.00,N,20/12/22,2023


## Cell 3 - Inspect columns and missing values

In [3]:
print("Columns:")
# Inspect the data to confirm the structure and values look reasonable.
print(df.columns.tolist())

null_summary = pd.DataFrame({
    # Handle missing values so later transformations and models do not fail.
    "null_count": df.isna().sum(),
    "null_pct": (df.isna().sum() / len(df)) * 100,
    "nunique": df.nunique(dropna=False)
}).sort_values("null_pct", ascending=False)

display(null_summary)

Columns:
['ID_NOTICE_CAN', 'TED_NOTICE_URL', 'YEAR', 'ID_TYPE', 'DT_DISPATCH', 'XSD_VERSION', 'CANCELLED', 'CORRECTIONS', 'B_MULTIPLE_CAE', 'CAE_NAME', 'CAE_NATIONALID', 'CAE_ADDRESS', 'CAE_TOWN', 'CAE_POSTAL_CODE', 'CAE_GPA_ANNEX', 'ISO_COUNTRY_CODE', 'ISO_COUNTRY_CODE_GPA', 'B_MULTIPLE_COUNTRY', 'ISO_COUNTRY_CODE_ALL', 'CAE_TYPE', 'EU_INST_CODE', 'MAIN_ACTIVITY', 'B_ON_BEHALF', 'B_INVOLVES_JOINT_PROCUREMENT', 'B_AWARDED_BY_CENTRAL_BODY', 'TYPE_OF_CONTRACT', 'TAL_LOCATION_NUTS', 'B_FRA_AGREEMENT', 'FRA_ESTIMATED', 'B_FRA_CONTRACT', 'B_DYN_PURCH_SYST', 'CPV', 'MAIN_CPV_CODE_GPA', 'ID_LOT', 'ADDITIONAL_CPVS', 'B_GPA', 'GPA_COVERAGE', 'LOTS_NUMBER', 'VALUE_EURO', 'VALUE_EURO_FIN_1', 'VALUE_EURO_FIN_2', 'B_EU_FUNDS', 'TOP_TYPE', 'B_ACCELERATED', 'OUT_OF_DIRECTIVES', 'CRIT_CODE', 'CRIT_PRICE_WEIGHT', 'CRIT_CRITERIA', 'CRIT_WEIGHTS', 'B_ELECTRONIC_AUCTION', 'NUMBER_AWARDS', 'ID_AWARD', 'ID_LOT_AWARDED', 'INFO_ON_NON_AWARD', 'INFO_UNPUBLISHED', 'B_AWARDED_TO_A_GROUP', 'WIN_NAME', 'WIN_NATION

,null_count,null_pct,nunique
MAIN_CPV_CODE_GPA,50000,100.000,1
GPA_COVERAGE,50000,100.000,1
ISO_COUNTRY_CODE_GPA,50000,100.000,1
CAE_GPA_ANNEX,50000,100.000,1
ISO_COUNTRY_CODE_ALL,49994,99.988,5
...,...,...,...
ISO_COUNTRY_CODE,0,0.000,32
INFO_UNPUBLISHED,0,0.000,2
NUMBER_AWARDS,0,0.000,177
OUT_OF_DIRECTIVES,0,0.000,2


## Cell 4 - Remove exact duplicate rows

In [4]:
# Filter the data to keep the records relevant for this step.
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]

print(f"Rows before duplicate removal: {before}")
print(f"Rows after duplicate removal:  {after}")
print(f"Exact duplicates removed:      {before - after}")

Rows before duplicate removal: 50000
Rows after duplicate removal:  50000
Exact duplicates removed:      0


## Cell 5 - Choose the target

In [5]:
TARGET = "VALUE_EURO"
print("Chosen target:", TARGET)

Chosen target: VALUE_EURO


## Cell 6 - Drop rows where target is missing or invalid

In [6]:
# Convert target to numeric if needed
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

before = len(df)

# Remove missing target rows
df = df[df[TARGET].notna()]

# Remove zero or negative values if present
df = df[df[TARGET] > 0]

after = len(df)

print(f"Rows before target filtering: {before}")
print(f"Rows after target filtering:  {after}")
print(f"Rows removed:                 {before - after}")

Rows before target filtering: 50000
Rows after target filtering:  46503
Rows removed:                 3497


## Cell 7 - Drop fully null, near-null, and constant columns

In [7]:
# Fully null columns
full_null_cols = [col for col in df.columns if df[col].isna().all()]

# Near-null columns (95%+ missing)
near_null_cols = [col for col in df.columns if df[col].isna().mean() >= 0.95]

# Constant columns
constant_cols = [col for col in df.columns if df[col].nunique(dropna=False) <= 1]

# Inspect the data to confirm the structure and values look reasonable.
print("Fully null columns:", full_null_cols)
print("Near-null columns:", near_null_cols)
print("Constant columns:", constant_cols)

drop_stage1 = sorted(set(full_null_cols + near_null_cols + constant_cols))
print("\nColumns to drop in stage 1:", drop_stage1)

df = df.drop(columns=drop_stage1, errors="ignore")

print("Shape after stage 1 drop:", df.shape)

Fully null columns: ['CAE_GPA_ANNEX', 'ISO_COUNTRY_CODE_GPA', 'MAIN_CPV_CODE_GPA', 'GPA_COVERAGE']
Near-null columns: ['CAE_GPA_ANNEX', 'ISO_COUNTRY_CODE_GPA', 'ISO_COUNTRY_CODE_ALL', 'EU_INST_CODE', 'MAIN_CPV_CODE_GPA', 'GPA_COVERAGE', 'B_ACCELERATED']
Constant columns: ['YEAR', 'CANCELLED', 'CAE_GPA_ANNEX', 'ISO_COUNTRY_CODE_GPA', 'MAIN_CPV_CODE_GPA', 'GPA_COVERAGE', 'source_year']

Columns to drop in stage 1: ['B_ACCELERATED', 'CAE_GPA_ANNEX', 'CANCELLED', 'EU_INST_CODE', 'GPA_COVERAGE', 'ISO_COUNTRY_CODE_ALL', 'ISO_COUNTRY_CODE_GPA', 'MAIN_CPV_CODE_GPA', 'YEAR', 'source_year']
Shape after stage 1 drop: (46503, 66)


## Cell 8 - Drop redundant columns

In [8]:
# Filter the data to keep the records relevant for this step.
redundant_cols = []

# Common TED redundancies seen earlier
for col in ["source_year", "VALUE_EURO_FIN_2"]:
    if col in df.columns:
        redundant_cols.append(col)

# Inspect the data to confirm the structure and values look reasonable.
print("Redundant columns to drop:", redundant_cols)

df = df.drop(columns=redundant_cols, errors="ignore")

print("Shape after redundant drop:", df.shape)

Redundant columns to drop: ['VALUE_EURO_FIN_2']
Shape after redundant drop: (46503, 65)


## Cell 9 - Drop irrelevant ID/admin columns

In [9]:
id_admin_cols = [
    "ID_NOTICE_CAN",
    "TED_NOTICE_URL",
    "ID_TYPE",
    "XSD_VERSION",
    "ID_LOT",
    "ID_AWARD",
    "ID_LOT_AWARDED",
    "CONTRACT_NUMBER",
    "CAE_NATIONALID",
    "CAE_ADDRESS",
    "CAE_POSTAL_CODE"
]

# Filter the data to keep the records relevant for this step.
existing_id_admin_cols = [col for col in id_admin_cols if col in df.columns]
# Inspect the data to confirm the structure and values look reasonable.
print("Dropping ID/admin columns:", existing_id_admin_cols)

df = df.drop(columns=existing_id_admin_cols, errors="ignore")

print("Shape after ID/admin drop:", df.shape)

Dropping ID/admin columns: ['ID_NOTICE_CAN', 'TED_NOTICE_URL', 'ID_TYPE', 'XSD_VERSION', 'ID_LOT', 'ID_AWARD', 'ID_LOT_AWARDED', 'CONTRACT_NUMBER', 'CAE_NATIONALID', 'CAE_ADDRESS', 'CAE_POSTAL_CODE']
Shape after ID/admin drop: (46503, 54)


## Cell 10 - Drop award-stage / leakage columns

In [10]:
leakage_cols = [
    "AWARD_VALUE_EURO",
    "AWARD_EST_VALUE_EURO",
    "AWARD_VALUE_EURO_FIN_1",
    "WIN_NAME",
    "WIN_NATIONALID",
    "WIN_ADDRESS",
    "WIN_TOWN",
    "WIN_POSTAL_CODE",
    "WIN_COUNTRY_CODE",
    "NUMBER_AWARDS",
    "B_AWARDED_TO_A_GROUP",
    "B_CONTRACTOR_SME",
    "B_SUBCONTRACTED",
    "DT_AWARD",
    "INFO_ON_NON_AWARD"
]

# Filter the data to keep the records relevant for this step.
existing_leakage_cols = [col for col in leakage_cols if col in df.columns]
# Inspect the data to confirm the structure and values look reasonable.
print("Dropping leakage columns:", existing_leakage_cols)

df = df.drop(columns=existing_leakage_cols, errors="ignore")

print("Shape after leakage drop:", df.shape)

Dropping leakage columns: ['AWARD_VALUE_EURO', 'AWARD_EST_VALUE_EURO', 'AWARD_VALUE_EURO_FIN_1', 'WIN_NAME', 'WIN_NATIONALID', 'WIN_ADDRESS', 'WIN_TOWN', 'WIN_POSTAL_CODE', 'WIN_COUNTRY_CODE', 'NUMBER_AWARDS', 'B_AWARDED_TO_A_GROUP', 'B_CONTRACTOR_SME', 'B_SUBCONTRACTED', 'DT_AWARD', 'INFO_ON_NON_AWARD']
Shape after leakage drop: (46503, 39)


## Cell 11 - Drop low-relevance procedural columns

In [11]:
procedural_cols = [
    "CORRECTIONS",
    "B_MULTIPLE_CAE",
    "B_MULTIPLE_COUNTRY",
    "B_ON_BEHALF",
    "B_INVOLVES_JOINT_PROCUREMENT",
    "B_AWARDED_BY_CENTRAL_BODY",
    "B_GPA",
    "INFO_UNPUBLISHED",
    "OUT_OF_DIRECTIVES",
    "B_ELECTRONIC_AUCTION",
    "B_DYN_PURCH_SYST",
    "B_FRA_AGREEMENT",
    "FRA_ESTIMATED",
    "B_FRA_CONTRACT",
    "B_EU_FUNDS",
    "VALUE_EURO_FIN_1"   # drop if using VALUE_EURO as target
]

# Filter the data to keep the records relevant for this step.
existing_procedural_cols = [col for col in procedural_cols if col in df.columns]
# Inspect the data to confirm the structure and values look reasonable.
print("Dropping procedural / low-value columns:", existing_procedural_cols)

df = df.drop(columns=existing_procedural_cols, errors="ignore")

print("Shape after procedural drop:", df.shape)

Dropping procedural / low-value columns: ['CORRECTIONS', 'B_MULTIPLE_CAE', 'B_MULTIPLE_COUNTRY', 'B_ON_BEHALF', 'B_INVOLVES_JOINT_PROCUREMENT', 'B_AWARDED_BY_CENTRAL_BODY', 'B_GPA', 'INFO_UNPUBLISHED', 'OUT_OF_DIRECTIVES', 'B_ELECTRONIC_AUCTION', 'B_DYN_PURCH_SYST', 'B_FRA_AGREEMENT', 'FRA_ESTIMATED', 'B_FRA_CONTRACT', 'B_EU_FUNDS', 'VALUE_EURO_FIN_1']
Shape after procedural drop: (46503, 23)


## Cell 12 - Keep only the core modelling columns

In [12]:
core_columns = [
    "TITLE",
    "CPV",
    "ADDITIONAL_CPVS",
    "TYPE_OF_CONTRACT",
    "TOP_TYPE",
    "MAIN_ACTIVITY",
    "CAE_TYPE",
    "ISO_COUNTRY_CODE",
    "TAL_LOCATION_NUTS",
    "LOTS_NUMBER",
    "DT_DISPATCH",
    TARGET
]

# Filter the data to keep the records relevant for this step.
existing_core_columns = [col for col in core_columns if col in df.columns]
# Define the model or model set that will be trained and compared.
df_model = df[existing_core_columns].copy()

# Inspect the data to confirm the structure and values look reasonable.
print("Final modelling columns:")
print(df_model.columns.tolist())

print("\nShape of modelling dataframe:", df_model.shape)
display(df_model.head())

Final modelling columns:
['TITLE', 'CPV', 'ADDITIONAL_CPVS', 'TYPE_OF_CONTRACT', 'TOP_TYPE', 'MAIN_ACTIVITY', 'CAE_TYPE', 'ISO_COUNTRY_CODE', 'TAL_LOCATION_NUTS', 'LOTS_NUMBER', 'DT_DISPATCH', 'VALUE_EURO']

Shape of modelling dataframe: (46503, 12)


,TITLE,CPV,ADDITIONAL_CPVS,TYPE_OF_CONTRACT,TOP_TYPE,MAIN_ACTIVITY,CAE_TYPE,ISO_COUNTRY_CODE,TAL_LOCATION_NUTS,LOTS_NUMBER,DT_DISPATCH,VALUE_EURO
0,NaN,85000000,NaN,S,NIC,Health,6,AT,AT,1.0,28/12/22,16257000.00
1,Suministro de diversos vehículos para el Parqu...,34144900,NaN,U,OPE,General public\services,3,ES,ES521,1.0,28/12/22,109000.00
2,Laboratórne zariadenia,38000000,42900000,U,OPE,Health,6,SK,SK,4.0,28/12/22,1942083.33
3,Zariadenia na spracovanie vzoriek,38000000,42900000,U,OPE,Health,6,SK,SK,4.0,28/12/22,1942083.33
4,Zariadenia pre potreby experimentálneho zverinca,38000000,42900000,U,OPE,Health,6,SK,SK,4.0,28/12/22,1942083.33


## Cell 13 - Convert dates and engineer time features

In [13]:
if "DT_DISPATCH" in df_model.columns:
    # First try day-first parsing, which is common in TED-style European dates
    df_model["DT_DISPATCH"] = pd.to_datetime(
        df_model["DT_DISPATCH"],
        errors="coerce",
        dayfirst=True
    )

    # Filter the data to keep the records relevant for this step.
    df_model["DISPATCH_YEAR"] = df_model["DT_DISPATCH"].dt.year
    df_model["DISPATCH_MONTH"] = df_model["DT_DISPATCH"].dt.month
    df_model["DISPATCH_QUARTER"] = df_model["DT_DISPATCH"].dt.quarter

    # Define the model or model set that will be trained and compared.
    df_model = df_model.drop(columns=["DT_DISPATCH"])

print("Columns after date engineering:")
# Inspect the data to confirm the structure and values look reasonable.
print(df_model.columns.tolist())

Columns after date engineering:
['TITLE', 'CPV', 'ADDITIONAL_CPVS', 'TYPE_OF_CONTRACT', 'TOP_TYPE', 'MAIN_ACTIVITY', 'CAE_TYPE', 'ISO_COUNTRY_CODE', 'TAL_LOCATION_NUTS', 'LOTS_NUMBER', 'VALUE_EURO', 'DISPATCH_YEAR', 'DISPATCH_MONTH', 'DISPATCH_QUARTER']


/tmp/ipykernel_31426/186342733.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_model["DT_DISPATCH"] = pd.to_datetime(


## Cell 14 - Handle missing values

In [14]:
# Fill text/categorical columns with "Unknown"
categorical_cols = df_model.select_dtypes(include=["object", "string"]).columns.tolist()

for col in categorical_cols:
    # Handle missing values so later transformations and models do not fail.
    df_model[col] = df_model[col].fillna("Unknown")

# Fill numeric columns with median, except target
numeric_cols = df_model.select_dtypes(include=[np.number]).columns.tolist()
# Filter the data to keep the records relevant for this step.
numeric_feature_cols = [col for col in numeric_cols if col != TARGET and col != "LOG_" + TARGET]

# Choose the columns that will be used as features or summary variables.
for col in numeric_feature_cols:
    df_model[col] = df_model[col].fillna(df_model[col].median())

print("Remaining missing values:")
# Inspect the data to confirm the structure and values look reasonable.
display(df_model.isna().sum().sort_values(ascending=False))

Remaining missing values:


TITLE                0
CPV                  0
ADDITIONAL_CPVS      0
TYPE_OF_CONTRACT     0
TOP_TYPE             0
MAIN_ACTIVITY        0
CAE_TYPE             0
ISO_COUNTRY_CODE     0
TAL_LOCATION_NUTS    0
LOTS_NUMBER          0
VALUE_EURO           0
DISPATCH_YEAR        0
DISPATCH_MONTH       0
DISPATCH_QUARTER     0
dtype: int64

## Cell 15 - Log transform the target

In [15]:
# Convert values into analysis-friendly numeric, date, or text formats.
df_model["LOG_" + TARGET] = np.log1p(df_model[TARGET])

# Inspect the data to confirm the structure and values look reasonable.
print(df_model[[TARGET, "LOG_" + TARGET]].head())

    VALUE_EURO  LOG_VALUE_EURO
0  16257000.00       16.604034
1    109000.00       11.599112
2   1942083.33       14.479272
3   1942083.33       14.479272
4   1942083.33       14.479272


## Cell 16 - Final inspection

In [16]:
# Inspect the data to confirm the structure and values look reasonable.
print("Final cleaned modelling dataset shape:", df_model.shape)
print("\nFinal columns:")
print(df_model.columns.tolist())

display(df_model.head())

Final cleaned modelling dataset shape: (46503, 15)

Final columns:
['TITLE', 'CPV', 'ADDITIONAL_CPVS', 'TYPE_OF_CONTRACT', 'TOP_TYPE', 'MAIN_ACTIVITY', 'CAE_TYPE', 'ISO_COUNTRY_CODE', 'TAL_LOCATION_NUTS', 'LOTS_NUMBER', 'VALUE_EURO', 'DISPATCH_YEAR', 'DISPATCH_MONTH', 'DISPATCH_QUARTER', 'LOG_VALUE_EURO']


,TITLE,CPV,ADDITIONAL_CPVS,TYPE_OF_CONTRACT,TOP_TYPE,MAIN_ACTIVITY,CAE_TYPE,ISO_COUNTRY_CODE,TAL_LOCATION_NUTS,LOTS_NUMBER,VALUE_EURO,DISPATCH_YEAR,DISPATCH_MONTH,DISPATCH_QUARTER,LOG_VALUE_EURO
0,Unknown,85000000,Unknown,S,NIC,Health,6,AT,AT,1.0,16257000.00,2022,12,4,16.604034
1,Suministro de diversos vehículos para el Parqu...,34144900,Unknown,U,OPE,General public\services,3,ES,ES521,1.0,109000.00,2022,12,4,11.599112
2,Laboratórne zariadenia,38000000,42900000,U,OPE,Health,6,SK,SK,4.0,1942083.33,2022,12,4,14.479272
3,Zariadenia na spracovanie vzoriek,38000000,42900000,U,OPE,Health,6,SK,SK,4.0,1942083.33,2022,12,4,14.479272
4,Zariadenia pre potreby experimentálneho zverinca,38000000,42900000,U,OPE,Health,6,SK,SK,4.0,1942083.33,2022,12,4,14.479272


## Cell 17 - Optional: save cleaned dataset

In [19]:
os.makedirs("../data/processed", exist_ok=True)
output_file = "../data/processed/ted_cleaned_for_cost_prediction.csv"
# Save the processed output so later notebooks or report sections can reuse it.

df_model.to_csv(output_file, index=False)

print(f"Saved cleaned dataset to: {output_file}")

Saved cleaned dataset to: ../data/processed/ted_cleaned_for_cost_prediction.csv
